# %% [markdown]
# # Phân tích câu hỏi trắc nghiệm theo CTT trên EdNet KT1
#
# Chạy các ô theo thứ tự (Shift+Enter). Ô 0 chứa mọi tham số. Cần: `pip install pandas numpy pyarrow tqdm`.
#
# | Ô | Việc làm |
# |---|---|
# | 0 | Cấu hình |
# | 1 | Đọc và làm sạch bảng câu hỏi |
# | 2 | Làm sạch học viên (đọc song song, theo lô, chạy tiếp được) |
# | 3 | Kiểm tra sau làm sạch, lấy lần làm đầu tiên |
# | 4 | Khối dữ liệu dày (Part 5) |
# | 5 | CTT trên ma trận thưa |
# | 6 | Chẩn đoán kết quả CTT |
# | 7 | Hàm CTT đầy đủ (D27%, rpbis, KR-20, phương án nhiễu) |
# | 8 | Kiểm chứng công thức bằng dữ liệu đề thi giả lập |
# | 9 | CTT đầy đủ trên EdNet (bài thi nhỏ mà nhiều người làm đủ mọi câu) |

In [36]:
# %% Ô 0. Cấu hình
import os
import json
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
 
import numpy as np
import pandas as pd
 
try:
    from tqdm.auto import tqdm
except ImportError:                      # không có tqdm thì vẫn chạy, chỉ không có thanh tiến trình
    def tqdm(x, **kwargs):
        return x
 
ROOT = Path(os.environ.get("EDNET_ROOT", r"C:\Datasets\EdNet"))
OUT = Path(os.environ.get("EDNET_OUT", r"C:\Datasets\processed"))
OUT.mkdir(parents=True, exist_ok=True)
 
N_FILES = int(os.environ.get("N_FILES", 50000))        # số học viên đọc (tăng dần được: các lô cũ được dùng lại)
SEED = 42
N_WORKERS = int(os.environ.get("N_WORKERS", 8))        # số luồng đọc file
CHUNK = int(os.environ.get("CHUNK", 1000))             # số học viên mỗi lô
 
# --- ngưỡng làm sạch (lựa chọn của nhóm, không phải chuẩn của EdNet: hãy ghi vào báo cáo) ---
VALID = {"a", "b", "c", "d"}          # đáp án hợp lệ
MIN_RESP = 20                         # người có ít hơn số lượt này bị loại
MIN_STRAIGHT, TOP_SHARE = 30, 0.90    # chọn cùng một đáp án > 90% lượt (với >= 30 lượt) -> loại
MAX_ELAPSED = 600_000                 # ms; chỉ ĐÁNH DẤU, không xóa
 
# --- khối dữ liệu cho CTT trên ma trận thưa ---
N_ITEMS = int(os.environ.get("N_ITEMS", 200))          # số câu Part 5 được làm nhiều nhất
MIN_ANS = int(os.environ.get("MIN_ANS", 50))           # người phải làm >= số câu này trong khối
MIN_ITEM_N = int(os.environ.get("MIN_ITEM_N", 100))    # câu cần >= số người này mới đáng tin
DENSE_MIN = int(os.environ.get("DENSE_MIN", 60))       # ô 6: chỉ giữ người làm >= số câu này
 
# --- ngưỡng phân loại câu hỏi theo bài báo gốc ---
P_LOW, P_HIGH, R_MIN = 0.3, 0.7, 0.2

In [37]:
# %% Ô 1. Đọc và làm sạch bảng câu hỏi
def clean_questions(q):
    q = q.copy()
    n0 = len(q)
    q["question_id"] = q["question_id"].astype(str).str.strip()
    q = q.drop_duplicates("question_id")
    q["correct_answer"] = q["correct_answer"].astype(str).str.strip().str.lower()
    q = q[q["correct_answer"].isin(VALID)]
    q["part"] = pd.to_numeric(q["part"], errors="coerce")
    print(f"Câu hỏi: {n0} -> {len(q)}")
    return q
 
 
q = clean_questions(pd.read_csv(ROOT / "contents" / "questions.csv"))
ans_map = q.set_index("question_id")["correct_answer"]      # đáp án đúng theo câu
part_map = q.set_index("question_id")["part"]               # phần thi theo câu
print(q["part"].value_counts().sort_index())

Câu hỏi: 13169 -> 13169
part
1     643
2    1662
3    1266
4    1158
5    5703
6    1335
7    1402
Name: count, dtype: int64


In [38]:
# %% Ô 2. Làm sạch học viên: đọc song song, làm sạch theo lô, lưu từng lô và chạy tiếp được
COLS = ["timestamp", "solving_id", "question_id", "user_answer", "elapsed_time"]
 
 
def list_files(folder):
    """Liệt kê 784 nghìn file rất chậm trên Windows -> chỉ làm một lần rồi lưu lại (xóa file cache nếu đổi dữ liệu)."""
    cache = OUT / "kt1_file_list.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8").splitlines()
    with os.scandir(folder) as it:
        names = sorted(e.path for e in it if e.name.endswith(".csv"))
    cache.write_text("\n".join(names), encoding="utf-8")
    return names
 
 
def read_raw(path):
    d = pd.read_csv(path, usecols=COLS)
    d["user_id"] = Path(path).stem
    return d
 
 
def clean_chunk(raw, n_files):
    """Làm sạch cho cả lô học viên cùng lúc (ít lệnh pandas hơn nhiều so với làm từng file)."""
    s = Counter()
    s["1_dong_goc"] = len(raw)
 
    d = raw.dropna(subset=["timestamp", "question_id", "user_answer"])
    s["2_sau_bo_thieu"] = len(d)
 
    d = d.assign(user_answer=d["user_answer"].astype(str).str.strip().str.lower())
    d = d[d["user_answer"].isin(VALID)]
    s["3_dap_an_hop_le"] = len(d)
 
    d = d[d["question_id"].isin(ans_map.index)]
    s["4_cau_co_trong_bang_cau_hoi"] = len(d)
 
    d = d.drop_duplicates(["user_id", "timestamp", "question_id"])
    s["5_sau_bo_trung"] = len(d)
 
    cnt = d.groupby("user_id").size()
    enough = cnt.index[cnt >= MIN_RESP]
    s["nguoi_bi_loai_qua_it_luot"] = n_files - len(enough)       # gồm cả người mất hết dòng
    d = d[d["user_id"].isin(enough)]
 
    cnt = d.groupby("user_id").size()
    top_share = d.groupby(["user_id", "user_answer"]).size().groupby("user_id").max() / cnt
    straight = cnt.index[(cnt >= MIN_STRAIGHT) & (top_share > TOP_SHARE)]
    s["nguoi_bi_loai_chon_mot_dap_an"] = len(straight)
    d = d[~d["user_id"].isin(straight)]
 
    d = d.sort_values(["user_id", "timestamp", "solving_id"], kind="mergesort")   # theo thời gian: để xác định lần làm đầu tiên
    d["attempt_no"] = d.groupby(["user_id", "question_id"]).cumcount() + 1       # lần làm thứ mấy của câu đó
    d["correct"] = (d["user_answer"] == d["question_id"].map(ans_map)).astype(int)
    d["bad_time"] = (d["elapsed_time"] <= 0) | (d["elapsed_time"] > MAX_ELAPSED)
    s["nguoi_giu_lai"] = d["user_id"].nunique()
    return d, s
 
 
files = list_files(ROOT / "KT1")
print("Số file KT1:", len(files))
# hoán vị cố định theo SEED: tăng N_FILES sau này chỉ xử lý thêm phần mới, các lô cũ được dùng lại
order = np.random.default_rng(SEED).permutation(len(files))
sample = [files[i] for i in order[:min(N_FILES, len(files))]]
 
clean_dir = OUT / "clean_all"
clean_dir.mkdir(exist_ok=True)
signature = dict(VALID=sorted(VALID), MIN_RESP=MIN_RESP, MIN_STRAIGHT=MIN_STRAIGHT, TOP_SHARE=TOP_SHARE,
                 MAX_ELAPSED=MAX_ELAPSED, SEED=SEED, CHUNK=CHUNK)
sig_path = clean_dir / "_params.json"
if not sig_path.exists() or json.loads(sig_path.read_text()) != signature:
    for old in list(clean_dir.glob("*.parquet")) + list(clean_dir.glob("*.json")):   # đổi ngưỡng -> làm lại từ đầu
        old.unlink()
    sig_path.write_text(json.dumps(signature), encoding="utf-8")
 
stats = Counter()
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    for i in tqdm(range(0, len(sample), CHUNK)):
        tag = f"{i // CHUNK:04d}"
        part, js = clean_dir / f"part_{tag}.parquet", clean_dir / f"stats_{tag}.json"
        if js.exists():                                        # lô này đã xong ở lần chạy trước
            stats.update(json.loads(js.read_text()))
            continue
        paths = sample[i:i + CHUNK]
        raw = pd.concat(list(pool.map(read_raw, paths)), ignore_index=True)
        d, s = clean_chunk(raw, len(paths))
        if len(d):
            d.to_parquet(part)
        js.write_text(json.dumps(dict(s)), encoding="utf-8")   # ghi sau cùng: có file này nghĩa là lô đã hoàn tất
        stats.update(s)
 
print("\n== Bảng đếm làm sạch ==")
for key_, v in sorted(stats.items()):
    print(f"{key_:34s}{v:>14,}")
(OUT / "clean_report.json").write_text(json.dumps(dict(stats), indent=2), encoding="utf-8")

Số file KT1: 784309

== Bảng đếm làm sạch ==
1_dong_goc                             6,001,769
2_sau_bo_thieu                         5,999,785
3_dap_an_hop_le                        5,999,785
4_cau_co_trong_bang_cau_hoi            5,999,785
5_sau_bo_trung                         5,999,456
nguoi_bi_loai_chon_mot_dap_an                  8
nguoi_bi_loai_qua_it_luot                 29,359
nguoi_giu_lai                             20,633


259

In [39]:
# %% Ô 3. Kiểm tra sau làm sạch và lấy lần làm đầu tiên (CTT chỉ dùng lần đầu của mỗi người với mỗi câu)
part_files = sorted(clean_dir.glob("part_*.parquet"))
if not part_files:
    raise SystemExit("Không còn người nào sau làm sạch: kiểm tra lại ngưỡng ở ô 0 và bảng đếm ở ô 2.")
 
# chỉ đọc các file part_*.parquet (thư mục còn có file stats_*.json nên không đọc cả thư mục)
clean = pd.concat([pd.read_parquet(f) for f in part_files], ignore_index=True)
assert clean["user_answer"].isin(VALID).all()
assert clean[["user_id", "question_id", "timestamp", "correct"]].notna().all().all()
assert clean.groupby("user_id")["timestamp"].apply(lambda s: s.is_monotonic_increasing).all()
print(clean.shape, "| số người:", clean["user_id"].nunique(),
      "| tỷ lệ đúng:", round(clean["correct"].mean(), 3),
      "| tỷ lệ bad_time:", round(clean["bad_time"].mean(), 3))
 
first = clean[clean["attempt_no"] == 1]
del clean                                                               # giải phóng RAM
first.to_parquet(OUT / "ednet_first_attempts.parquet")
first5 = first[first["question_id"].map(part_map) == 5]                 # Part 5: câu đứng độc lập
first5.to_parquet(OUT / "ednet_first_part5.parquet")
print("lần đầu:", first.shape, "| Part 5:", first5.shape)

(5777204, 9) | số người: 20633 | tỷ lệ đúng: 0.661 | tỷ lệ bad_time: 0.003
lần đầu: (4825038, 9) | Part 5: (1981962, 9)


In [40]:
# %% Ô 4. Khối dữ liệu dày (Part 5, lần làm đầu tiên): hàng = người, cột = câu, ô = 1 (đúng) / 0 (sai) / NaN (chưa làm)
def build_block(df, n_items=N_ITEMS, min_ans=MIN_ANS):
    top = df["question_id"].value_counts().head(n_items).index
    b = df[df["question_id"].isin(top)]
    cnt = b["user_id"].value_counts()
    b = b[b["user_id"].isin(cnt[cnt >= min_ans].index)]
    return b.pivot(index="user_id", columns="question_id", values="correct")
 
 
R = build_block(first5)
print("R:", R.shape, "| tỷ lệ ô có dữ liệu:", round(R.notna().mean().mean(), 3))
print("số câu mỗi người (trung vị):", R.notna().sum(axis=1).median())
print("số người mỗi câu (nhỏ nhất / trung vị):", R.notna().sum().min(), "/", R.notna().sum().median())
R.to_parquet(OUT / "R_part5.parquet")

R: (2680, 200) | tỷ lệ ô có dữ liệu: 0.5
số câu mỗi người (trung vị): 89.0
số người mỗi câu (nhỏ nhất / trung vị): 741 / 1289.5


In [41]:
# %% Ô 5. CTT trên ma trận thưa: độ khó p, tương quan câu - phần còn lại, phân loại theo bài báo
def classify(p, r):
    if P_LOW <= p <= P_HIGH and r > R_MIN:
        return "phu hop"
    issues = []
    if p < P_LOW:
        issues.append("qua kho")
    if p > P_HIGH:
        issues.append("qua de")
    if pd.isna(r) or r <= R_MIN:
        issues.append("phan cach thap")
    return ", ".join(issues)
 
 
def ctt_sparse(R, min_n=MIN_ITEM_N):
    n_ans, sum_ans = R.notna().sum(axis=1), R.sum(axis=1)
 
    def rest_corr(col):
        # điểm "phần còn lại" theo tỷ lệ đúng, vì mỗi người làm số câu khác nhau
        rest = (sum_ans - col.fillna(0)) / (n_ans - col.notna().astype(int))
        return col.corr(rest)
 
    t = pd.DataFrame({"n": R.notna().sum(), "p": R.mean(), "r_rest": R.apply(rest_corr)})
    t["class"] = [classify(p, r) for p, r in zip(t.p, t.r_rest)]
    t.loc[t.n < min_n, "class"] = "it du lieu (n < %d)" % min_n       # cờ này ưu tiên: kết quả chưa đáng tin
    return t
 
 
ctt = ctt_sparse(R)
print(ctt.describe().round(3))
print(ctt["class"].value_counts())
ctt.to_csv(OUT / "ctt_part5.csv")
# Lưu ý: D27% và KR-20 cần ma trận đầy đủ (mọi người làm cùng bộ câu) nên không tính trên ma trận thưa này (xem ô 7 và 9).

              n        p   r_rest
count   200.000  200.000  200.000
mean   1339.160    0.529    0.193
std     322.438    0.156    0.083
min     741.000    0.141   -0.037
25%    1084.750    0.444    0.132
50%    1289.500    0.538    0.194
75%    1565.250    0.648    0.253
max    2486.000    0.860    0.408
class
phu hop                    79
phan cach thap             74
qua de, phan cach thap     18
qua kho, phan cach thap    15
qua de                      9
qua kho                     5
Name: count, dtype: int64


In [42]:
# %% Ô 6. Chẩn đoán: cờ "phân cách thấp" là tín hiệu thật hay do ma trận thưa?
# 1) độ nhạy với ngưỡng
for t in (0.10, 0.15, 0.20, 0.25, 0.30):
    print(f"r_rest > {t:.2f}: {(ctt.r_rest > t).mean():.1%} số câu")
 
# 2) r_rest có phụ thuộc độ khó không? (câu rất dễ/khó có r thấp theo cơ chế toán học)
print("tương quan p ~ r_rest:", round(ctt["p"].corr(ctt["r_rest"]), 3))
bins = pd.cut(ctt["p"], [0, 0.3, 0.45, 0.6, 0.7, 1.0])
print(ctt.groupby(bins, observed=True)["r_rest"].agg(["count", "median"]).round(3))
 
# 3) chỉ dùng người làm nhiều câu để điểm "phần còn lại" bớt nhiễu
dense = R.loc[R.notna().sum(axis=1) >= DENSE_MIN]
if len(dense) > 0:
    c2 = ctt_sparse(dense, min_n=MIN_ITEM_N)
    print("người:", dense.shape[0], "| r_rest trung vị:", round(c2["r_rest"].median(), 3),
          "| tỷ lệ câu phù hợp:", round((c2["class"] == "phu hop").mean(), 3))
 
# 4) câu có r_rest rất thấp hoặc âm: nghi đáp án sai khóa hoặc câu mơ hồ
low = ctt.sort_values("r_rest").head(10).join(q.set_index("question_id")[["part", "correct_answer"]])
print(low[["n", "p", "r_rest", "correct_answer"]].round(3))

r_rest > 0.10: 87.0% số câu
r_rest > 0.15: 69.5% số câu
r_rest > 0.20: 46.5% số câu
r_rest > 0.25: 26.0% số câu
r_rest > 0.30: 10.5% số câu
tương quan p ~ r_rest: 0.055
             count  median
p                         
(0.0, 0.3]      20   0.129
(0.3, 0.45]     35   0.228
(0.45, 0.6]     77   0.209
(0.6, 0.7]      41   0.200
(0.7, 1.0]      27   0.167
người: 2222 | r_rest trung vị: 0.196 | tỷ lệ câu phù hợp: 0.41
                n      p  r_rest correct_answer
question_id                                    
q6525        1615  0.282  -0.037              c
q4543        1086  0.541  -0.008              a
q6507        1019  0.220  -0.006              d
q4317        1680  0.252   0.011              c
q6127        1324  0.591   0.015              a
q5667        1063  0.849   0.021              b
q6442        1494  0.372   0.029              d
q5659        1095  0.233   0.033              b
q3698        1565  0.429   0.035              d
q6213         965  0.367   0.057              b


In [43]:
# %% Ô 7. Hàm CTT đầy đủ cho một kỳ thi (mọi thí sinh làm cùng bộ câu)
# A: đáp án thô (hàng = thí sinh, cột = câu); key: đáp án đúng theo câu (Series)
def score(A, key):
    return (A == key).astype(int)
 
 
def kr20(R):
    k, p = R.shape[1], R.mean()
    return (k / (k - 1)) * (1 - (p * (1 - p)).sum() / R.sum(axis=1).var(ddof=0))
 
 
def _groups(R, frac=0.27):
    total = R.sum(axis=1).sort_values(kind="mergesort")
    n = int(round(len(R) * frac))
    return total.index[:n], total.index[-n:]                            # nhóm thấp, nhóm cao
 
 
def d27(R):
    low, high = _groups(R)
    return R.loc[high].mean() - R.loc[low].mean()
 
 
def r_pbis(R, corrected=False):
    total = R.sum(axis=1)
    return R.apply(lambda c: c.corr(total - c if corrected else total))
 
 
def distractors(A, R, key, options, min_share=0.05):
    low, high = _groups(R)
    rows = []
    for item in A.columns:
        for opt in options:
            pa = (A[item] == opt).mean()
            ph = (A.loc[high, item] == opt).mean()
            pl = (A.loc[low, item] == opt).mean()
            is_key = key[item] == opt
            flag = ""
            if is_key and ph <= pl:
                flag = "dap an dung khong phan biet duoc nhom gioi"
            if not is_key and pa < min_share:                           # min_share là quy ước, có thể chỉnh
                flag = "it nguoi chon"
            if not is_key and ph > pl:
                flag = "thu hut nhom gioi hon nhom yeu"
            rows.append(dict(item=item, option=opt, is_key=is_key, p_all=pa, p_high=ph, p_low=pl, flag=flag))
    return pd.DataFrame(rows)
 
 
def analyze(A, key, corrected=False, options=None):
    R = score(A, key)
    if options is None:                                                  # EdNet: a-d (thường); giả lập: A-D (hoa)
        options = sorted(set(key.unique()) | set(np.unique(A.to_numpy())))
    items = pd.DataFrame({"p": R.mean(), "d27": d27(R),
                          "r_pbis": r_pbis(R, corrected), "r_pbis_rest": r_pbis(R, True)})
    r = items["r_pbis_rest"] if corrected else items["r_pbis"]
    items["class"] = [classify(p, rr) for p, rr in zip(items.p, r)]
    return {"kr20": kr20(R), "items": items, "distractors": distractors(A, R, key, options)}

In [44]:
# %% Ô 8. Kiểm chứng công thức bằng dữ liệu đề thi giả lập (biết trước độ khó và độ phân cách thật)
def simulate_exam(n_items=120, n_persons=2000, guess=0.2, seed=42, options="ABCD"):
    rng = np.random.default_rng(seed)
    theta = rng.normal(0, 1, n_persons)                                 # năng lực thí sinh
    a = rng.lognormal(0, 0.3, n_items)                                  # độ phân cách thật
    b = rng.normal(0, 1, n_items)                                       # độ khó thật
    p = guess + (1 - guess) / (1 + np.exp(-a * (theta[:, None] - b)))
    correct = rng.random(p.shape) < p
    key = rng.choice(list(options), n_items)
    A = np.empty(p.shape, dtype="<U1")
    for j in range(n_items):
        wrong = [o for o in options if o != key[j]]
        w = rng.dirichlet(np.ones(len(wrong)))                          # mỗi câu một kiểu phương án nhiễu
        A[:, j] = np.where(correct[:, j], key[j], rng.choice(wrong, size=n_persons, p=w))
    items = [f"q{j + 1:03d}" for j in range(n_items)]
    users = [f"s{i + 1:04d}" for i in range(n_persons)]
    return (pd.DataFrame(A, index=users, columns=items),
            pd.Series(key, index=items),
            pd.DataFrame({"a": a, "b": b}, index=items))
 
 
A_sim, key_sim, truth = simulate_exam()
res_sim = analyze(A_sim, key_sim)
print("KR-20:", round(res_sim["kr20"], 3))
print(res_sim["items"]["class"].value_counts())
chk = res_sim["items"].join(truth)
print(chk[["p", "r_pbis", "a", "b"]].corr().round(2))                   # kỳ vọng: p ~ b rất âm, r_pbis ~ a dương
A_sim.to_csv(OUT / "exam_answers.csv")
key_sim.to_csv(OUT / "exam_key.csv", header=["key"])
res_sim["items"].to_csv(OUT / "exam_ctt_items.csv")
res_sim["distractors"].to_csv(OUT / "exam_distractors.csv", index=False)

KR-20: 0.928
class
phu hop                    84
qua de                     28
phan cach thap              4
qua kho, phan cach thap     4
Name: count, dtype: int64
           p  r_pbis     a     b
p       1.00    0.52  0.03 -0.98
r_pbis  0.52    1.00  0.62 -0.52
a       0.03    0.62  1.00 -0.01
b      -0.98   -0.52 -0.01  1.00


In [45]:
# %% Ô 9. CTT đầy đủ trên EdNet: tìm "bài thi nhỏ" mà nhiều người làm đủ mọi câu (dùng đáp án thật a-d)
MIN_COMPLETE = int(os.environ.get("MIN_COMPLETE", 300))     # số người tối thiểu làm đủ mọi câu (quy ước, chỉnh được)
POOL = int(os.environ.get("POOL", 100))                     # xét POOL câu Part 5 được làm nhiều nhất
 
top_items = first5["question_id"].value_counts().index[:POOL]
A_all = (first5[first5["question_id"].isin(top_items)]
         .pivot(index="user_id", columns="question_id", values="user_answer")[list(top_items)])
answered = A_all.notna()
curve = {K: int(answered.iloc[:, :K].all(axis=1).sum()) for K in range(5, len(top_items) + 1, 5)}
print("K câu -> số người làm đủ K câu đầu (xếp theo độ phổ biến):")
for K_, n_ in curve.items():
    print(f"  {K_:>3} câu: {n_:>6,} người")
 
ok = [K_ for K_, n_ in curve.items() if n_ >= MIN_COMPLETE]
if not ok:
    print(f"Chưa có K nào có >= {MIN_COMPLETE} người làm đủ: tăng N_FILES (ô 0) hoặc hạ MIN_COMPLETE.")
else:
    K_ITEMS = int(os.environ.get("K_ITEMS", max(ok)))        # mặc định: bộ dài nhất vẫn còn đủ người
    A_ed = A_all.iloc[:, :K_ITEMS].dropna()
    key_ed = ans_map[A_ed.columns]
    print(f"\nBài thi nhỏ: {A_ed.shape[1]} câu x {A_ed.shape[0]} người (làm đủ mọi câu)")
    res_ed = analyze(A_ed, key_ed)
    print("KR-20:", round(res_ed["kr20"], 3))
    print(res_ed["items"]["class"].value_counts())
    print("Phương án nhiễu bị gắn cờ:", (res_ed["distractors"]["flag"] != "").sum(),
          "/", len(res_ed["distractors"]))
    res_ed["items"].to_csv(OUT / "ednet_ctt_items.csv")
    res_ed["distractors"].to_csv(OUT / "ednet_ctt_distractors.csv", index=False)
 

K câu -> số người làm đủ K câu đầu (xếp theo độ phổ biến):
    5 câu: 10,580 người
   10 câu:    993 người
   15 câu:    531 người
   20 câu:    283 người
   25 câu:    218 người
   30 câu:    183 người
   35 câu:    162 người
   40 câu:    129 người
   45 câu:    125 người
   50 câu:    116 người
   55 câu:    115 người
   60 câu:     78 người
   65 câu:     75 người
   70 câu:     63 người
   75 câu:     63 người
   80 câu:     45 người
   85 câu:     41 người
   90 câu:     41 người
   95 câu:     39 người
  100 câu:     39 người

Bài thi nhỏ: 15 câu x 531 người (làm đủ mọi câu)
KR-20: 0.366
class
phu hop                    10
qua kho                     2
qua kho, phan cach thap     2
qua de                      1
Name: count, dtype: int64
Phương án nhiễu bị gắn cờ: 8 / 60
